<a href="https://colab.research.google.com/github/Vidyaratnam-G/Quantitative_Analysis/blob/main/Stock_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.subplots as sp
import plotly.graph_objects as go

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ---------- Load ----------
jepq = pd.read_excel('/content/drive/MyDrive/Stock_Analysis/Datasets/JEPQ_Daily.xlsx')
qqq  = pd.read_excel('/content/drive/MyDrive/Stock_Analysis/Datasets/QQQ_Daily.xlsx')
vxn  = pd.read_excel('/content/drive/MyDrive/Stock_Analysis/Datasets/VXN_Daily.xlsx')
rfr = pd.read_excel('/content/drive/MyDrive/Stock_Analysis/Datasets/RiskFreeRate_Daily.xlsx')

In [ ]:
jepq.head(5)

,Date,Open,High,Low,Close,Adj Close,Volume
0,NaT,NaN,NaN,NaN,Close price adjusted for splits.,Adjusted close price adjusted for splits and d...,NaN
1,2025-12-26,59.39,59.42,59.25,59.34,59.34,4258900.0
2,2025-12-24,59.16,59.31,59.12,59.29,59.29,2746900.0
3,2025-12-23,58.87,59.17,58.80,59.15,59.15,3954500.0
4,2025-12-22,59,59.00,58.74,58.89,58.89,3729200.0


 The clean_price_df() function prepares raw daily price data for analysis. It first standardizes column names by stripping extra whitespace and removes embedded metadata rows that do not contain valid dates. The Date column is then parsed into a datetime format and set as the index to enable time-series operations. Prices are coerced to numeric values to guard against hidden text entries, and any invalid or missing observations are dropped. The resulting output is a clean, date-indexed DataFrame with a single standardized price column.

In [ ]:
def clean_price_df(df):
    """
    Clean and standardize raw daily price data.

    This function:
    - Removes metadata rows embedded in the data
    - Standardizes column names by stripping whitespace
    - Parses and sets the Date column as a datetime index
    - Automatically detects the Adjusted Close price column
    - Converts prices to numeric values
    - Drops missing or invalid observations

    The output is a DataFrame indexed by date with a single column named
    'price'.
    """
    df = df.copy()

    # Strip column names
    df.columns = df.columns.str.strip()

    # Drop metadata row
    df = df[df["Date"].notna()]

    # Parse date
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.set_index("Date")

    # Find adjusted close column automatically
    adj_close_candidates = [c for c in df.columns if "Adj Close" in c]

    if len(adj_close_candidates) == 0:
        raise ValueError("No Adjusted Close column found.")

    adj_close_col = adj_close_candidates[0]

    # Keep only adjusted close
    df = df[[adj_close_col]]
    df[adj_close_col] = pd.to_numeric(df[adj_close_col], errors="coerce")

    # Final cleanup
    df = df.dropna()

    return df.rename(columns={adj_close_col: "price"})

In [ ]:
jepq = clean_price_df(jepq)
qqq  = clean_price_df(qqq)
vxn  = clean_price_df(vxn)
rfr = clean_price_df(rfr)

In [ ]:
jepq.head(5)

,price
Date,
2025-12-26,59.34
2025-12-24,59.29
2025-12-23,59.15
2025-12-22,58.89
2025-12-19,58.62


In [ ]:
# ---------- Merge ----------
prices = pd.concat(
    [jepq.rename(columns={"price":"JEPQ"}),
     qqq.rename(columns={"price":"QQQ"}),
     vxn.rename(columns={"price":"VXN"}),
     rfr.rename(columns={"price":"RFR"})],
    axis=1
).dropna()

In [ ]:
prices.head(5)

,JEPQ,QQQ,VXN,RFR
Date,,,,
2025-12-26,59.34,623.89,17.38,4.136
2025-12-24,59.29,623.93,17.20,4.136
2025-12-23,59.15,622.11,17.47,4.169
2025-12-22,58.89,619.21,17.85,4.169
2025-12-19,58.62,616.26,18.52,4.151


The returns dataframe computes daily returns for JEPQ and QQQ based on their adjusted closing prices. It first selects the relevant price columns from the prices DataFrame, then applies pct_change() to calculate the percentage change from one trading day to the next, which represents each asset’s daily return. Because the first observation has no prior day to compare against, it results in a missing value and is removed using dropna(). The resulting DataFrame is indexed by date and contains daily return series for each asset, expressed as fractional returns (e.g., 0.01 corresponds to a 1% daily gain)

In [ ]:
returns = prices[["JEPQ", "QQQ", "VXN"]].pct_change().dropna()

In [ ]:
returns.head(5)

,JEPQ,QQQ,VXN
Date,,,
2025-12-24,-0.000843,0.000064,-0.010357
2025-12-23,-0.002361,-0.002917,0.015698
2025-12-22,-0.004396,-0.004662,0.021752
2025-12-19,-0.004585,-0.004764,0.037535
2025-12-18,-0.012112,-0.012868,0.118790


In [ ]:
# Create 3 vertically stacked subplots with a shared x-axis
fig = sp.make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    subplot_titles=[
        "JEPQ Daily Return",
        "QQQ Daily Return",
        "VXN Daily Change"
    ]
)

# JEPQ subplot
fig.add_trace(
    go.Scatter(
        x=returns.index,
        y=returns["JEPQ"],
        mode="lines",
        name="JEPQ",
        hovertemplate="Date: %{x}<br>Daily Change: %{y:.2%}<extra></extra>"
    ),
    row=1, col=1
)

# QQQ subplot
fig.add_trace(
    go.Scatter(
        x=returns.index,
        y=returns["QQQ"],
        mode="lines",
        name="QQQ",
        hovertemplate="Date: %{x}<br>Daily Change: %{y:.2%}<extra></extra>"
    ),
    row=2, col=1
)

# VXN subplot
fig.add_trace(
    go.Scatter(
        x=returns.index,
        y=returns["VXN"],
        mode="lines",
        name="VXN",
        hovertemplate="Date: %{x}<br>Daily Change: %{y:.2%}<extra></extra>"
    ),
    row=3, col=1
)

# Layout tweaks
fig.update_layout(
    height=900,
    title="Daily Percentage Change: JEPQ, QQQ, and VXN",
    hovermode="x unified"
)

fig.show()

This analysis measures the relationship between daily returns of JEPQ and QQQ and daily percentage changes in the VXN index, which captures shocks to market volatility rather than its absolute level. All series are converted to daily returns using percentage changes, and correlations are computed between each ETF’s returns and contemporaneous changes in VXN. A negative correlation indicates that asset returns tend to decline on days when volatility increases, while a weaker or less negative relationship suggests greater resilience to volatility shocks.

In [ ]:
# Compute correlations
vxn_return_corr = pd.DataFrame({
    "JEPQ": returns["JEPQ"].corr(returns["VXN"]),
    "QQQ":  returns["QQQ"].corr(returns["VXN"])
}, index=["Correlation with VXN Daily Change"])

vxn_return_corr

,JEPQ,QQQ
Correlation with VXN Daily Change,-0.701745,-0.700617


The results show a strong negative correlation between daily changes in the VXN index and same-day returns for both JEPQ and QQQ, with correlations of approximately −0.70 for each. This indicates that on days when market volatility increases sharply, both ETFs tend to experience significant negative returns, while decreases in volatility are generally associated with positive returns. The similarity of the correlations suggests that, despite JEPQ’s income-oriented and volatility-selling structure, it remains highly exposed to short-term volatility shocks in much the same way as QQQ. In other words, selling volatility through option premiums does not meaningfully insulate JEPQ from sudden increases in market uncertainty; instead, both strategies tend to move risk-off when volatility spikes, even though their longer-term return profiles may differ.

This code computes key distributional statistics for the daily return series of JEPQ, QQQ, and VXN. Specifically, it calculates the mean (average daily return), standard deviation (a measure of volatility), skewness (the asymmetry of the return distribution), and kurtosis (the degree of tailedness).

In [ ]:
distribution_stats = pd.DataFrame({
    "Mean": returns.mean(),
    "Std Dev": returns.std(),
    "Skew": returns.skew(),
    "Kurtosis": returns.kurtosis()
})

distribution_stats

,Mean,Std Dev,Skew,Kurtosis
JEPQ,-0.000540,0.010661,0.035575,10.061008
QQQ,-0.000621,0.014271,0.032942,5.228124
VXN,0.002086,0.052783,-0.406037,2.839592


The distributional statistics reveal meaningful differences in the behavior of JEPQ, QQQ, and VXN. Both JEPQ and QQQ show slightly negative average daily returns, with JEPQ at -0.054% and QQQ at -0.062%, indicating modest drift downward over the period analyzed. JEPQ exhibits lower volatility (standard deviation of 1.07%) compared to QQQ (1.43%), suggesting that its returns fluctuate less day-to-day. Interestingly, JEPQ has a much higher kurtosis (10.06) than QQQ (5.23), indicating that JEPQ experiences more extreme tail events—sharp gains or losses—despite its lower overall volatility. Both ETFs show minimal skewness, implying their return distributions are relatively symmetric. In contrast, VXN, representing market volatility, displays a positive mean return (0.21%) and dramatically higher standard deviation (5.28%), consistent with its nature as a volatility measure that spikes during market stress. Its negative skew (-0.41) suggests occasional large upward spikes in volatility, while its lower kurtosis (2.84) indicates that extreme volatility moves are less frequent than the tail events observed in JEPQ. Overall, these results highlight that while JEPQ may appear less volatile than QQQ on average, it is prone to more extreme return outliers, whereas VXN captures broader market turbulence with both higher variance and asymmetric spikes.

# TODO:

Question: Should I put money in QQQ or JEPQ? Do cumulative return instead of daily return. Do sum of daily returns. Plot daily volatility (VXN) and cumulative returns. Why doesn't JEPQ beat QQQ, despite including high performance QQQ stocks only and dividends as well? QQQ is better in days with high volatility (as the upside is not capped). Maybe JEPQ is better in days with low volatility?

In [ ]:
# ---------- Cumulative Returns & VXN Level Plot ----------
# Calculate cumulative returns
cum_returns = (1 + returns[["JEPQ", "QQQ"]]).cumprod() - 1

In [ ]:
cum_returns.head(5)

,JEPQ,QQQ
Date,,
2025-12-24,-0.000843,0.000064
2025-12-23,-0.003202,-0.002853
2025-12-22,-0.007583,-0.007501
2025-12-19,-0.012133,-0.012230
2025-12-18,-0.024098,-0.024940


In [ ]:
# Extract the absolute VXN level for the same dates
vxn_level = prices.loc[returns.index, "VXN"]

In [ ]:
# Create a 2-row subplot: Top for Cumulative Returns, Bottom for Volatility
fig_cum = sp.make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.7, 0.3],
    subplot_titles=["Cumulative Return: JEPQ vs QQQ", "VXN Index Level (Volatility)"]
)

# JEPQ Cumulative Return
fig_cum.add_trace(
    go.Scatter(x=cum_returns.index, y=cum_returns["JEPQ"], name="JEPQ Cum Return", mode="lines"),
    row=1, col=1
)

# QQQ Cumulative Return
fig_cum.add_trace(
    go.Scatter(x=cum_returns.index, y=cum_returns["QQQ"], name="QQQ Cum Return", mode="lines"),
    row=1, col=1
)

# VXN Level
fig_cum.add_trace(
    go.Scatter(x=vxn_level.index, y=vxn_level, name="VXN Level", mode="lines", line=dict(color="red")),
    row=2, col=1
)

fig_cum.update_layout(
    height=800,
    title="Cumulative Performance vs. Market Volatility",
    hovermode="x unified"
)
fig_cum.update_yaxes(tickformat=".2%", row=1, col=1)

fig_cum.show()

In [ ]:
# ---------- Volatility Regime Analysis ----------
# Create a new dataframe for analysis
analysis_df = returns[["JEPQ", "QQQ"]].copy()
analysis_df["VXN_Level"] = prices.loc[returns.index, "VXN"]

In [ ]:
# Define regimes based on the median VXN level in the dataset
median_vxn = analysis_df["VXN_Level"].median()
analysis_df["Vol_Regime"] = np.where(analysis_df["VXN_Level"] > median_vxn, "High Volatility", "Low Volatility")

In [ ]:
# Calculate average daily returns, then annualize them
regime_returns = analysis_df.groupby("Vol_Regime")[["JEPQ", "QQQ"]].mean() * 252

In [ ]:
print("Annualized Expected Return by Volatility Regime:")
display(regime_returns.map(lambda x: f"{x:.2%}"))

Annualized Expected Return by Volatility Regime:


,JEPQ,QQQ
Vol_Regime,,
High Volatility,-11.92%,-14.40%
Low Volatility,-15.27%,-16.92%


In [ ]:
# ---------- Up Market vs Down Market Capture ----------
# Define market direction based on the broader index (QQQ)
analysis_df["Market_Dir"] = np.where(analysis_df["QQQ"] > 0, "Up Day", "Down Day")

In [ ]:
# Calculate the average return on Up days vs Down days
capture_stats = analysis_df.groupby("Market_Dir")[["JEPQ", "QQQ"]].mean()

In [ ]:
# Calculate capture ratios
up_capture = capture_stats.loc["Up Day", "JEPQ"] / capture_stats.loc["Up Day", "QQQ"]
down_capture = capture_stats.loc["Down Day", "JEPQ"] / capture_stats.loc["Down Day", "QQQ"]

In [ ]:
print("Average Daily Return on Up vs. Down Days:")
display(capture_stats.map(lambda x: f"{x:.2%}"))

Average Daily Return on Up vs. Down Days:


,JEPQ,QQQ
Market_Dir,,
Down Day,-0.68%,-0.98%
Up Day,0.72%,1.07%


In [ ]:
print(f"\nUpside Capture Ratio: {up_capture:.1%}")
print(f"Downside Capture Ratio: {down_capture:.1%}")


Upside Capture Ratio: 66.8%
Downside Capture Ratio: 69.1%


# Redone TODO:

In [ ]:
# ---------- Sort Data Chronologically ----------
# Sort the index ascending so the oldest date is at the top (iloc[0])
sorted_prices = prices.sort_index(ascending=True)

In [ ]:
sorted_prices.head(5)

,JEPQ,QQQ,VXN,RFR
Date,,,,
2022-05-04,34.38,321.79,31.98,2.917
2022-05-05,33.57,305.58,37.19,3.066
2022-05-06,33.12,301.92,37.40,3.123
2022-05-09,32.11,290.11,41.43,3.079
2022-05-10,32.34,293.63,40.08,2.993


In [ ]:
# ---------- Cumulative Returns & VXN Level Plot ----------
# Calculate cumulative returns directly from prices (Day X / Day 1) - 1
cum_returns = (sorted_prices[["JEPQ", "QQQ"]] / sorted_prices[["JEPQ", "QQQ"]].iloc[0]) - 1

In [ ]:
cum_returns.head(5)

,JEPQ,QQQ
Date,,
2022-05-04,0.000000,0.000000
2022-05-05,-0.023560,-0.050374
2022-05-06,-0.036649,-0.061748
2022-05-09,-0.066027,-0.098449
2022-05-10,-0.059337,-0.087510


In [ ]:
cum_returns.tail(5)

,JEPQ,QQQ
Date,,
2025-12-19,0.705061,0.915100
2025-12-22,0.712914,0.924267
2025-12-23,0.720477,0.933279
2025-12-24,0.724549,0.938935
2025-12-26,0.726003,0.938811


In [ ]:
# Extract the absolute VXN level for the same dates
vxn_level = sorted_prices["VXN"]

In [ ]:
vxn_level.head(5)

,VXN
Date,
2022-05-04,31.98
2022-05-05,37.19
2022-05-06,37.40
2022-05-09,41.43
2022-05-10,40.08


In [ ]:
fig_cum = sp.make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.7, 0.3],
    subplot_titles=["Cumulative Return: JEPQ vs QQQ", "VXN Index Level (Volatility)"]
)

# JEPQ Cumulative Return
fig_cum.add_trace(
    go.Scatter(x=cum_returns.index, y=cum_returns["JEPQ"], name="JEPQ Cum Return", mode="lines"),
    row=1, col=1
)

# QQQ Cumulative Return
fig_cum.add_trace(
    go.Scatter(x=cum_returns.index, y=cum_returns["QQQ"], name="QQQ Cum Return", mode="lines"),
    row=1, col=1
)

# VXN Level
fig_cum.add_trace(
    go.Scatter(x=vxn_level.index, y=vxn_level, name="VXN Level", mode="lines", line=dict(color="red")),
    row=2, col=1
)

fig_cum.update_layout(
    height=800,
    title="Cumulative Performance vs. Market Volatility",
    hovermode="x unified"
)
fig_cum.update_yaxes(tickformat=".2%", row=1, col=1)

fig_cum.show()

In [ ]:
# ---------- Volatility Regime Analysis (Cumulative) ----------
# Create a new dataframe for analysis based on cumulative returns
analysis_df = cum_returns.copy()
analysis_df["VXN_Level"] = sorted_prices["VXN"]

In [ ]:
# Define regimes based on the median VXN level in the dataset
median_vxn = analysis_df["VXN_Level"].median()
analysis_df["Vol_Regime"] = np.where(analysis_df["VXN_Level"] > median_vxn, "High Volatility", "Low Volatility")

print(median_vxn)

21.365000000000002


In [ ]:
# Calculate average cumulative returns
regime_returns = analysis_df.groupby("Vol_Regime")[["JEPQ", "QQQ"]].mean()

In [ ]:
print("Average Cumulative Total Return by Current Volatility Regime:")
display(regime_returns.map(lambda x: f"{x:.2%}"))

Average Cumulative Total Return by Current Volatility Regime:


,JEPQ,QQQ
Vol_Regime,,
High Volatility,12.79%,16.09%
Low Volatility,34.03%,44.31%


Note: High volatility results in around the same difference between percentage returns (scaled) between JEPQ and QQQ. Only difference is that returns are lower in higher volatility periods.

In [ ]:
# ---------- Up Market vs Down Market Capture (Cumulative) ----------
# Define market direction based on whether QQQ is overall up or down since Day 1
analysis_df["Market_Dir"] = np.where(analysis_df["QQQ"] > 0, "Since-Inception Up", "Since-Inception Down")

In [ ]:
# Calculate the average cumulative return on overall Up vs Down periods
capture_stats = analysis_df.groupby("Market_Dir")[["JEPQ", "QQQ"]].mean()

In [ ]:
print("\nAverage Cumulative Return on Overall Up vs. Down Periods:")
display(capture_stats.map(lambda x: f"{x:.2%}"))


Average Cumulative Return on Overall Up vs. Down Periods:


,JEPQ,QQQ
Market_Dir,,
Since-Inception Down,-7.39%,-10.31%
Since-Inception Up,35.29%,45.83%


Note: JEPQ losses in down markets are less than QQQ losses in down markets, percentage wise. Theory: Maybe because JEPQ contains top QQQ stocks, so losses are minimised in JEPQ.

In [ ]:
# Calculate cumulative capture ratios
up_capture = capture_stats.loc["Since-Inception Up", "JEPQ"] / capture_stats.loc["Since-Inception Up", "QQQ"]
print(f"\nCumulative Upside Capture Ratio: {up_capture:.1%}")

down_capture = capture_stats.loc["Since-Inception Down", "JEPQ"] / capture_stats.loc["Since-Inception Down", "QQQ"]
print(f"Cumulative Downside Capture Ratio: {down_capture:.1%}")


Cumulative Upside Capture Ratio: 77.0%
Cumulative Downside Capture Ratio: 71.6%


Overall, JEPQ maybe safer? Captures more upside of QQQ than downside of QQQ, so fewer sharp spikes in either direction, money wise.

# Smoothing

In [ ]:
# Smooth the cumulative returns using a Centered Rolling Mean
smoothing_window = 63
smoothed_qqq = cum_returns["QQQ"].rolling(window=smoothing_window, center=True).mean()

In [ ]:
# Identify local maxima and minima on the smoothed curve
is_max = (smoothed_qqq > smoothed_qqq.shift(1)) & (smoothed_qqq > smoothed_qqq.shift(-1))
is_min = (smoothed_qqq < smoothed_qqq.shift(1)) & (smoothed_qqq < smoothed_qqq.shift(-1))

In [ ]:
# Extract the specific dates/timestamps of these turning points
peak_dates = cum_returns.index[is_max]
trough_dates = cum_returns.index[is_min]

In [ ]:
peak_dates

DatetimeIndex(['2022-07-14', '2022-08-08', '2023-08-07', '2023-08-10',
               '2024-03-05', '2024-03-11', '2024-03-14', '2024-06-20',
               '2024-07-22', '2024-07-29', '2024-08-15', '2024-08-26',
               '2025-01-10', '2025-01-14'],
              dtype='datetime64[ns]', name='Date', freq=None)

In [ ]:
# Create a new Plotly figure
fig_smooth = go.Figure()

# 1. Plot the raw QQQ returns (faded so it doesn't distract from the trend)
fig_smooth.add_trace(
    go.Scatter(
        x=cum_returns.index,
        y=cum_returns["QQQ"],
        name="QQQ Raw",
        mode="lines",
        line=dict(color="blue", width=1),
        opacity=0.3  # Faded to show noise behind the trend
    )
)

# Plot the raw JEPQ returns (faded)
fig_smooth.add_trace(
    go.Scatter(
        x=cum_returns.index,
        y=cum_returns["JEPQ"],
        name="JEPQ Raw",
        mode="lines",
        line=dict(color="purple", width=1),
        opacity=0.3
    )
)

# Plot the smoothed QQQ trendline (Bold)
fig_smooth.add_trace(
    go.Scatter(
        x=smoothed_qqq.index,
        y=smoothed_qqq,
        name=f"QQQ Smoothed ({smoothing_window}-Day)",
        mode="lines",
        line=dict(color="blue", width=3)
    )
)

# Add Markers for Local Maxima
fig_smooth.add_trace(
    go.Scatter(
        x=peak_dates,
        y=smoothed_qqq.loc[peak_dates],
        name="Macro Peaks (Start of Downward Trend)",
        mode="markers",
        marker=dict(color="red", size=10, symbol="triangle-down"),
    )
)

# Add Markers for Local Minima
fig_smooth.add_trace(
    go.Scatter(
        x=trough_dates,
        y=smoothed_qqq.loc[trough_dates],
        name="Macro Troughs (Start of Upward Trend)",
        mode="markers",
        marker=dict(color="green", size=10, symbol="triangle-up"),
    )
)

fig_smooth.update_layout(
    height=600,
    title="Cumulative Returns: Raw Noise vs. Smoothed Macro Trends",
    hovermode="x unified",
    yaxis_title="Cumulative Return",
    xaxis_title="Date",
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

# Format the y-axis as percentages
fig_smooth.update_yaxes(tickformat=".2%")

# Display the interactive chart
fig_smooth.show()

In [ ]:
# Combine into a single sorted timeline of trend changes
# We also include the very first and last dates to capture the entire series
all_turning_dates = sorted(list(peak_dates) + list(trough_dates))
all_turning_dates = [cum_returns.index[0]] + all_turning_dates + [cum_returns.index[-1]]
# Remove duplicates if any turning point overlaps with boundary dates
all_turning_dates = sorted(list(set(all_turning_dates)))

In [ ]:
# Step through each segment and compute slopes on the raw data
slope_records = []
for i in range(len(all_turning_dates) - 1):
    start_date = all_turning_dates[i]
    end_date = all_turning_dates[i+1]

    # Count the number of trading days in this specific period
    start_idx = cum_returns.index.get_loc(start_date)
    end_idx = cum_returns.index.get_loc(end_date)
    duration_days = end_idx - start_idx

    # Skip extremely short segments that might occur at boundaries
    if duration_days < 21:
        continue

    # Calculate total change (Rise) from the RAW cumulative returns
    qqq_rise = cum_returns.loc[end_date, "QQQ"] - cum_returns.loc[start_date, "QQQ"]
    jepq_rise = cum_returns.loc[end_date, "JEPQ"] - cum_returns.loc[start_date, "JEPQ"]

    # Calculate Slope = Rise / Run (Daily rate of return change during this phase)
    qqq_slope = qqq_rise / duration_days
    jepq_slope = jepq_rise / duration_days

    # Label the regime based on market direction
    regime = "Market Expansion (Up)" if qqq_rise > 0 else "Market Contraction (Down)"

    slope_records.append({
        "Period": f"{start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}",
        "Regime": regime,
        "Duration (Days)": duration_days,
        "QQQ Daily Slope": f"{qqq_slope:.4%}",
        "JEPQ Daily Slope": f"{jepq_slope:.4%}",
        "Performance Ratio (JEPQ/QQQ)": f"{(jepq_slope / qqq_slope):.2f}" if qqq_slope != 0 else "N/A"
    })

In [ ]:
# Convert to DataFrame
df_analysis = pd.DataFrame(slope_records)
print("\n--- Slope Comparison across Dynamically Detected Market Phases ---")
print(df_analysis.to_string(index=False))


--- Slope Comparison across Dynamically Detected Market Phases ---
                  Period                    Regime  Duration (Days) QQQ Daily Slope JEPQ Daily Slope Performance Ratio (JEPQ/QQQ)
2022-05-04 to 2022-06-21 Market Contraction (Down)               32        -0.4549%         -0.3227%                         0.71
2022-08-08 to 2022-11-21 Market Contraction (Down)               74        -0.1574%         -0.1049%                         0.67
2022-11-21 to 2023-08-07     Market Expansion (Up)              176         0.1649%          0.1177%                         0.71
2023-08-10 to 2023-09-22 Market Contraction (Down)               30        -0.1037%         -0.0533%                         0.51
2023-09-22 to 2024-03-05     Market Expansion (Up)              112         0.2169%          0.1745%                         0.80
2024-03-15 to 2024-06-20     Market Expansion (Up)               66         0.2244%          0.1349%                         0.60
2024-08-28 to 2025-01-